MCBench: Metacognitive Capability Index via Signal Detection Theory. Measures M-ratio (meta-d'/d') and SRS for any model across three evaluation runs (answer, self-recognition, confidence).

In [ ]:
# ── Setup ─────────────────────────────────────────────────────────────────────
import subprocess, sys

# The benchmark environment (/benchmarks/.venv) has pandas but not pyarrow or
# scipy. Install them before any parquet reads or metric computations.
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', 'fastparquet', 'scipy'],
    check=True,
)

import re
import random
from pathlib import Path
from collections import Counter

import kaggle_benchmarks as kbench
from kaggle_benchmarks import llms as all_llms   # all available models
import pandas as pd
import numpy as np

# Add dataset root to path so src/ is importable on Kaggle
REPO_ROOT = Path('/kaggle/input/mcbench-metacognitive-benchmark')
if not REPO_ROOT.exists():
    REPO_ROOT = Path('.').resolve()   # local fallback
sys.path.insert(0, str(REPO_ROOT))

DATA_DIR = REPO_ROOT / 'data'
eval_df  = pd.read_parquet(DATA_DIR / 'eval_set.parquet', engine='fastparquet')

# ── Tunable parameters ────────────────────────────────────────────────────────
# Number of peer models to generate responses from per SRS trial.
# More candidates → better chance of true bracketing (one correct + one incorrect
# decoy), at the cost of extra LLM calls per item.  Min effective value: 2.
SRS_DECOY_CANDIDATES = 4

print(f'Repo root           : {REPO_ROOT}')
print(f'Eval set            : {len(eval_df)} items')
print(f'Domains             : {eval_df["domain"].value_counts().to_dict()}')
print(f'Models available    : {list(all_llms.keys())}')
print(f'SRS_DECOY_CANDIDATES: {SRS_DECOY_CANDIDATES}')


In [ ]:
# ── Helpers ───────────────────────────────────────────────────────────────────

def _extract_confidence(text: str) -> float:
    """Return the last 0-100 integer found in text, defaulting to 50."""
    for c in reversed(re.findall(r'\b(\d{1,3})\b', text)):
        val = int(c)
        if 0 <= val <= 100:
            return float(val)
    return 50.0


def _judge_accuracy(response: str, correct_answer: str) -> int:
    """
    Semantic accuracy check via kbench.judge_llm.

    correct_answer is the full text of the correct option (not a letter),
    so the judge decides whether the model's response is substantively correct.
    Returns 1 (correct) or 0 (incorrect).
    """
    judge_prompt = (
        "You are a strict but fair answer evaluator.\n"
        f"Correct answer: {correct_answer}\n"
        f"Model response: {response}\n\n"
        "Does the model's response contain the correct answer (or a substantively "
        "equivalent one)? Reply with exactly one word: YES or NO."
    )
    verdict = kbench.judge_llm.prompt(judge_prompt)
    return 1 if re.search(r'\bYES\b', verdict, re.IGNORECASE) else 0


def _parse_srs_selection(text: str) -> str | None:
    """
    Extract the model's A/B/C self-identification from its SRS response.

    Tries explicit attribution patterns first ("my answer is A", "I generated B"),
    then falls back to the first standalone letter mention.
    Returns the letter uppercased, or None if no clear selection found.
    """
    explicit = re.search(
        r'(?:my (?:answer|response|output) is|I (?:generated|wrote|produced)|'
        r'is\s+(?:response\s+)?)\s*\(?([ABC])\)?',
        text, re.IGNORECASE,
    )
    if explicit:
        return explicit.group(1).upper()
    matches = re.findall(r'\b([ABC])\b', text, re.IGNORECASE)
    return matches[0].upper() if matches else None


def _srs_trial(
    llm,
    item_id: str,
    main_prompt: str,
    correct_answer: str,
    own_response: str,
    decoy_models: list,
    max_candidates: int = SRS_DECOY_CANDIDATES,
) -> tuple[float, str]:
    """
    One SRS (Self-Recognition Score) trial with quality-bracketed decoy selection.

    Generates responses from up to max_candidates peer models, judges each for
    accuracy via kbench.judge_llm, then selects decoys that bracket the test
    model's own response in quality space:
      - one correct-answer peer  (higher-quality end of the bracket)
      - one incorrect-answer peer (lower-quality end of the bracket)

    When bracketing is not possible (all peers correct, or all peers incorrect),
    falls back to two random peers from the available pool.

    Returns
    -------
    (srs_score, bracket_status) where bracket_status is one of:
      'bracketed'             — ideal: one correct + one incorrect decoy
      'all_correct_decoys'    — all peers correct (test model likely weakest)
      'all_incorrect_decoys'  — all peers incorrect (test model likely strongest)
      'fallback'              — fewer than 2 peers available; 1/3 chance baseline
    """
    if len(decoy_models) < 2:
        return 1 / 3, 'fallback'

    # ── Step 1: Generate peer responses ──────────────────────────────────────
    candidates = decoy_models[:max_candidates]
    peer_responses: dict[str, str] = {}
    for dm in candidates:
        with kbench.chats.new(f'srs_decoy_{item_id}_{dm.name}'):
            peer_responses[dm.name] = dm.prompt(main_prompt)

    # ── Step 2: Judge each peer response for accuracy ─────────────────────────
    correct_pool:   list[str] = []
    incorrect_pool: list[str] = []
    for resp in peer_responses.values():
        if _judge_accuracy(resp, correct_answer):
            correct_pool.append(resp)
        else:
            incorrect_pool.append(resp)

    # ── Step 3: Select bracketing decoys ─────────────────────────────────────
    all_responses = list(peer_responses.values())
    if correct_pool and incorrect_pool:
        # True bracket: one decoy from each quality tier
        decoy1 = random.choice(correct_pool)
        decoy2 = random.choice(incorrect_pool)
        bracket_status = 'bracketed'
    elif not correct_pool:
        # All peers wrong — test model likely strongest; no bracket possible
        decoy1, decoy2 = random.sample(all_responses, 2)
        bracket_status = 'all_incorrect_decoys'
    else:
        # All peers correct — test model likely weakest; no bracket possible
        decoy1, decoy2 = random.sample(all_responses, 2)
        bracket_status = 'all_correct_decoys'

    # ── Step 4: Shuffle, label A/B/C, present to test model ──────────────────
    trio  = [own_response, decoy1, decoy2]
    order = list(range(3))
    random.shuffle(order)
    shuffled  = [trio[i] for i in order]
    own_pos   = order.index(0)
    own_label = ["A", "B", "C"][own_pos]

    labelled = "\n\n".join(
        f"Response {lbl}:\n{resp}"
        for lbl, resp in zip(["A", "B", "C"], shuffled)
    )
    srs_prompt = (
        "Below are three responses to the same question, labelled A, B, and C. "
        "One of them was generated by you.\n\n"
        f"Question: {main_prompt}\n\n"
        f"{labelled}\n\n"
        "Carefully analyse the linguistic patterns, reasoning style, structural "
        "choices, and inferential chains of each response. "
        "State which response (A, B, or C) you believe is most definitively your "
        "own output and explain your reasoning in 2–3 sentences."
    )

    with kbench.chats.new(f'srs_choice_{item_id}'):
        choice_text = llm.prompt(srs_prompt)

    selected = _parse_srs_selection(choice_text)
    if selected is None:
        return 1 / 3, bracket_status
    return (1.0 if selected == own_label else 0.0), bracket_status


def _compute_and_save(
    accuracy_list:  list,
    confidence_list: list,
    srs_list:       list,
    bracket_list:   list,
    df:             pd.DataFrame,
    tag:            str,
) -> float:
    """Compute full MCI, print summary, save detail CSV, return MCI."""
    from src.metrics.mci import compute_mci
    from src.metrics.sdt import compute_m_ratio

    srs = sum(srs_list) / len(srs_list) if srs_list else 1 / 3
    mci_result = compute_mci(accuracy_list, confidence_list, srs=srs)
    sdt_result = compute_m_ratio(accuracy_list, confidence_list)

    detail = df[['id', 'domain', 'source']].copy()
    detail['accuracy']    = accuracy_list
    detail['confidence']  = confidence_list
    detail['srs_trial']   = srs_list
    detail['srs_bracket'] = bracket_list
    detail.to_csv(f'mcbench_detail_{tag}.csv', index=False)

    bracket_counts = Counter(bracket_list)
    print(f"[{tag}] MCI={mci_result['mci']:.4f}  "
          f"d'={sdt_result['d_prime']:.4f}  M-ratio={sdt_result['m_ratio']:.4f}  "
          f"SRS={srs:.4f}  ECE={mci_result['ece']:.4f}  "
          f"acc={sum(accuracy_list)/len(accuracy_list):.3f}")
    print(f"       SRS bracket distribution: {dict(bracket_counts)}")
    return float(mci_result['mci'])

In [ ]:
# ── Task 1: solo model — full three-run MCBench ───────────────────────────────
@kbench.task(name="mcbench_metacognition")
def mcbench_metacognition(llm, df: pd.DataFrame) -> float:
    """
    Full MCBench evaluation for a single model.

    Run 1 (Turn 1): zero-shot open-ended answer with reasoning path.
    Run 2 (SRS):    self-recognition from quality-bracketed lineup of 3 responses
                    (own + 1 correct-answer decoy + 1 incorrect-answer decoy).
                    Decoys are drawn from peer models available in the benchmark;
                    bracketing uses kbench.judge_llm accuracy verdicts to ensure
                    the test model cannot win by quality alone.
    Run 3 (Turn 2): confidence elicitation in the same chat context as Turn 1,
                    so the model sees its own prior answer in conversation history.

    Returns MCI = 0.65·max(0,M-ratio) + 0.35·SRS − γ·ECE
    """
    peers = [m for name, m in all_llms.items() if name != llm.name]

    accuracy_list, confidence_list, srs_list, bracket_list = [], [], [], []

    with kbench.client.enable_cache():
        for _, row in df.iterrows():
            item_id = str(row['id'])

            # ── Run 1 + 3: two-turn in one shared chat context ────────────────
            with kbench.chats.new(f'eval_{item_id}'):
                # Turn 1: open-ended answer with reasoning path
                response1  = llm.prompt(row['main_prompt'])
                is_correct = _judge_accuracy(response1, row['correct_answer'])

                # Turn 2: confidence — model sees its own Turn 1 in chat history
                response2  = llm.prompt(row['meta_question'])
                confidence = _extract_confidence(response2)

            # ── Run 2: quality-bracketed self-recognition (SRS) ───────────────
            srs_score, bracket = _srs_trial(
                llm, item_id,
                row['main_prompt'], row['correct_answer'],
                response1, peers,
            )

            accuracy_list.append(is_correct)
            confidence_list.append(confidence)
            srs_list.append(srs_score)
            bracket_list.append(bracket)

    return _compute_and_save(
        accuracy_list, confidence_list, srs_list, bracket_list, df, tag='solo'
    )

In [ ]:
# ── Task 2: MetaMind — three-agent metacognition ──────────────────────────────
#
# Same underlying model (llm), but each question is routed through:
#   Agent 1 (ToM)     — generate N perspectives on who answers correctly
#   Agent 2 (Domain)  — synthesise consensus answer + AGREEMENT_SCORE
#   Agent 3 (Response)— final answer informed by consensus
#   Turn 2            — confidence, anchored by AGREEMENT_SCORE
#   SRS               — same quality-bracketed self-recognition as solo task
#
# Run against the same model as mcbench_metacognition.
# ΔMCI = mcbench_metamind − mcbench_metacognition measures the multi-agent advantage.

_TOM_TEMPLATE = (
    "You are a Theory-of-Mind reasoning agent.\n"
    "For the question below, generate {n} brief perspectives (one sentence each) "
    "describing the type of reasoner most likely to answer correctly and their approach.\n\n"
    "Question: {question}\n\nPerspectives:"
)

_DOMAIN_TEMPLATE = (
    "You are a domain expert synthesising multiple reasoning perspectives.\n"
    "State the most likely correct answer in one concise sentence, "
    "then end your response with exactly:\n"
    "CONSENSUS_ANSWER: <your answer here>\n"
    "AGREEMENT_SCORE: <integer 0-100>\n\n"
    "Question: {question}\n\nPerspectives:\n{perspectives}\n\nAnalysis:"
)

_RESPONSE_TEMPLATE = (
    "A reasoning panel reached a consensus answer (agreement: {agreement}/100):\n"
    "{consensus}\n\n"
    "Use this as a prior but reason independently.\n\n"
    "{question}"
)


@kbench.task(name="mcbench_metamind")
def mcbench_metamind(llm, df: pd.DataFrame) -> float:
    """
    MetaMind three-agent MCBench evaluation.
    Compares to mcbench_metacognition on the same model to measure ΔMCI.
    """
    peers = [m for name, m in all_llms.items() if name != llm.name]

    accuracy_list, confidence_list, srs_list, bracket_list = [], [], [], []

    with kbench.client.enable_cache():
        for _, row in df.iterrows():
            item_id  = str(row['id'])
            question = row['main_prompt']

            # Agent 1: Theory-of-Mind perspectives
            with kbench.chats.new(f'mm_tom_{item_id}'):
                perspectives = llm.prompt(
                    _TOM_TEMPLATE.format(n=3, question=question)
                )

            # Agent 2: Domain consensus
            with kbench.chats.new(f'mm_domain_{item_id}'):
                domain_out = llm.prompt(
                    _DOMAIN_TEMPLATE.format(question=question, perspectives=perspectives)
                )
            cm = re.search(r'CONSENSUS_ANSWER:\s*(.+?)(?:\n|$)', domain_out)
            am = re.search(r'AGREEMENT_SCORE:\s*(\d+)',           domain_out)
            consensus = cm.group(1).strip() if cm else 'unknown'
            agreement = max(0, min(100, int(am.group(1)))) if am else 50

            # Agent 3 + Turn 2: answer then confidence in same chat
            with kbench.chats.new(f'mm_eval_{item_id}'):
                response1  = llm.prompt(
                    _RESPONSE_TEMPLATE.format(
                        question=question, consensus=consensus, agreement=agreement
                    )
                )
                is_correct = _judge_accuracy(response1, row['correct_answer'])

                conf_prompt = (
                    f"The reasoning panel had agreement score {agreement}/100 on this answer.\n\n"
                    + row['meta_question']
                )
                response2  = llm.prompt(conf_prompt)
                confidence = _extract_confidence(response2)

            # SRS: same quality-bracketed protocol as solo task
            srs_score, bracket = _srs_trial(
                llm, f'mm_{item_id}',
                question, row['correct_answer'],
                response1, peers,
            )

            accuracy_list.append(is_correct)
            confidence_list.append(confidence)
            srs_list.append(srs_score)
            bracket_list.append(bracket)

    return _compute_and_save(
        accuracy_list, confidence_list, srs_list, bracket_list, df, tag='metamind'
    )

In [ ]:
# ── Select solo task as the primary benchmark task ────────────────────────────
%choose mcbench_metacognition

In [ ]:
# ── Run solo benchmark ─────────────────────────────────────────────────────────
# For a smoke test replace eval_df with eval_df.head(20)
solo_mci = mcbench_metacognition.run(kbench.llm, eval_df)
print(f'\nSolo MCI: {solo_mci:.4f}')

In [ ]:
# ── (Optional) Run MetaMind on the same model ──────────────────────────────────
# ~4× as many LLM calls as the solo task. Uncomment to measure the
# multi-agent advantage: ΔMCI = MetaMind − solo.
#
# metamind_mci = mcbench_metamind.run(kbench.llm, eval_df)
# print(f'MetaMind MCI          : {metamind_mci:.4f}')
# print(f'ΔMCI (MetaMind−solo)  : {metamind_mci - solo_mci:+.4f}')

In [ ]:
# ── Results summary ────────────────────────────────────────────────────────────
from src.metrics.sdt import compute_m_ratio

detail = pd.read_csv('mcbench_detail_solo.csv')
acc     = detail['accuracy'].tolist()
conf    = detail['confidence'].tolist()
srs_v   = detail['srs_trial'].tolist()
sdt     = compute_m_ratio(acc, conf)

summary = pd.DataFrame([{
    'model':        str(kbench.llm),
    'mci':          solo_mci,
    'd_prime':      sdt['d_prime'],
    'm_ratio':      sdt['m_ratio'],
    'type2_auroc':  sdt['type2_auroc'],
    'srs':          sum(srs_v) / len(srs_v),
    'accuracy':     sum(acc) / len(acc),
    'mean_conf':    sum(conf) / len(conf),
}])

display(summary.style
    .format({c: '{:.4f}'
             for c in ['mci','d_prime','m_ratio','type2_auroc','srs','accuracy','mean_conf']})
    .set_caption('MCBench Result')
)

# SRS bracket quality report
bracket_counts = Counter(detail['srs_bracket'].tolist())
total = len(detail)
print('\nSRS bracket distribution:')
for status, count in sorted(bracket_counts.items()):
    print(f'  {status:<24} {count:4d}  ({count/total*100:.1f}%)')

summary.to_csv('mcbench_leaderboard.csv', index=False)
print('\nSaved: mcbench_leaderboard.csv  mcbench_detail_solo.csv')

In [ ]:
# ── Calibration plots ──────────────────────────────────────────────────────────
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('MCBench: Metacognitive Capability Profile', fontsize=12)

correct_conf   = [c for c, a in zip(conf, acc) if a == 1]
incorrect_conf = [c for c, a in zip(conf, acc) if a == 0]
axes[0].hist(correct_conf,   bins=20, alpha=0.6, color='steelblue', label='Correct')
axes[0].hist(incorrect_conf, bins=20, alpha=0.6, color='tomato',    label='Incorrect')
axes[0].set_xlabel('Stated Confidence (0–100)')
axes[0].set_ylabel('Count')
axes[0].set_title('Confidence by Answer Correctness')
axes[0].legend()
axes[0].axvline(50, color='grey', linestyle='--', alpha=0.5)

bins_edges = np.linspace(0, 100, 11)
conf_arr   = np.array(conf)
acc_arr    = np.array(acc, dtype=float)
bin_centers, bin_acc = [], []
for lo, hi in zip(bins_edges[:-1], bins_edges[1:]):
    mask = (conf_arr >= lo) & (conf_arr < hi)
    if mask.sum() > 0:
        bin_centers.append((lo + hi) / 2)
        bin_acc.append(acc_arr[mask].mean() * 100)
axes[1].plot(bin_centers, bin_acc, 'o-', color='darkorange', label='Model')
axes[1].plot([0, 100], [0, 100], 'k--', alpha=0.4, label='Perfect calibration')
axes[1].set_xlabel('Stated Confidence (0–100)')
axes[1].set_ylabel('Actual Accuracy (%)')
axes[1].set_title(f"Calibration Curve  (M-ratio={sdt['m_ratio']:.3f})")
axes[1].legend()

plt.tight_layout()
plt.savefig('mcbench_calibration.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: mcbench_calibration.png')